In [8]:
import pandas as pd
import numpy as np
clientFile = "../../data/clientes.csv"
raw = pd.read_csv(clientFile)

print(f"{raw.shape[0]:,} rows x {raw.shape[1]} feature columns, indexed by RecordID")
raw.head()

1,143 rows x 13 feature columns, indexed by RecordID


,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
1,770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
2,827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
3,494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
4,429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015


---

### 2. Data Cleaning: Dropping the Fully Empty Column
We identified that the variable `customer_type` has 0 non-missing values out of 1,143 customers, so it is a completely empty column. Because it provides no predictive signal, it was removed from the customer dataset, and the cleaned table was saved for the following steps.

In [9]:
outputFile = "../../data/clientesDataset/clientes_clean.csv"

df_clean = raw.copy()

# 1. Remove the fully empty column
df_clean = df_clean.drop(columns=["customer_type"])

df_clean.to_csv(outputFile, index=False)

print(f"Dataset cleaned saved successfully to: {outputFile}")

# 2. Verification of the removal of the column
print(raw.shape, "->", df_clean.shape)

Dataset cleaned saved successfully to: ../../data/clientesDataset/clientes_clean.csv
(1143, 13) -> (1143, 12)


In [10]:
# A column distinct on more than half its rows identifies the row, not a
# property of it.
id_like = [c for c in raw.columns if raw[c].nunique() > 0.5 * len(raw)]
print(id_like)


['customer_id', 'postal_code', 'address', 'company_name']


---

### 2.2. Feature Type Separation & Exclusions
To prepare the customer table for the merge with `train.csv`/`test.csv` and for feature encoding, we divide its attributes into distinct functional groups:

1. **Excluded Features (identifiers):**
   * The customer key (`customer_id`) is kept in the table because it is the join key with `train.csv`/`test.csv`, but it is excluded from the predictors to prevent memorization of individual customers (*overfitting*).
   * Columns that are distinct on more than half of the rows (`company_name`, `address`, `postal_code`) identify the customer instead of describing it, so they are also excluded.
   * The empty column `customer_type` was already removed in the previous step.
   * This table has no target variable and no raw date strings: the target (`repeat_purchase_90d`) and `purchase_date` live in `train.csv`.

2. **Numeric Features (`numeric`):**
   * Dynamically selected numerical indicators. In this table this is only `customer_since_year`, which can later be turned into a customer seniority feature and scaled.

3. **Categorical Features (`categorical`):**
   * Dynamically identified non-numeric attributes (the geographic hierarchy: `country_or_region`, `commercial_region`, `island`, `municipality`, `city`, `postal_area`, `street`), designated for missing value imputation and categorical encoding (One-Hot for low cardinality / Frequency Encoding for high cardinality).

In [13]:
# Identifiers that MUST NOT be used for model training
# (customer_id stays in the table as the join key with train.csv/test.csv)
key = "customer_id"
id_like = [c for c in df_clean.columns if df_clean[c].nunique() > 0.5 * len(df_clean)]
identifiers = [key] + [c for c in id_like if c != key]

# Automatic separation between numerical and categorical variables
numeric = [
    c for c in df_clean.columns
    if c not in identifiers and pd.api.types.is_numeric_dtype(df_clean[c])
]

categorical = [
    c for c in df_clean.columns
    if c not in identifiers + numeric
]

# Validation Output
print(f"Total features: {df_clean.shape[1]} | Numeric: {len(numeric)} | Categorical: {len(categorical)}")
#print(df_clean[categorical].nunique().sort_values(ascending=False))

Total features: 12 | Numeric: 1 | Categorical: 7


**Conclusions:** Explicitly separating the customer key and raw identifiers (`customer_id`, `company_name`, `address`, `postal_code`) from the numerical and categorical features ensures strict adherence to machine learning best practices. `customer_id` is kept in the table only as the join key with `train.csv`/`test.csv`, while the remaining near-unique columns are excluded because they identify a customer rather than describe it. By isolating these non-predictive attributes before model training, we prevent the model from memorizing individual customers, avoid dimensionality inflation from high-cardinality identifiers, and guarantee that only valid, analytically sound predictors (`customer_since_year` and the geographic hierarchy) enter the feature matrix.

In [14]:
# Number of distinct levels per categorical feature, to choose the encoding
# (One-Hot for low cardinality / Frequency Encoding for high cardinality)
levels = raw[categorical].nunique().sort_values(ascending=False)
levels

street               511
city                 417
postal_area          271
municipality         183
commercial_region     37
country_or_region     11
island                10
dtype: int64

**Conclusions:** The cardinality of the categorical features varies widely, so a single encoding strategy is not suitable for all of them:

* **Low cardinality** (`island` with 10 levels and `country_or_region` with 11): these can be safely One-Hot encoded without a large increase in dimensionality. However, `country_or_region` is highly unbalanced (about 94% of customers belong to CT10), so most of its levels are rare and it may be more informative as a binary flag.
* **Medium cardinality** (`commercial_region` with 37 levels): One-Hot encoding is still feasible if rare regions are grouped into an `"Other"` category. Otherwise, Frequency Encoding is a more compact alternative.
* **High cardinality** (`municipality` with 183, `postal_area` with 271, `city` with 417 and `street` with 511 levels): One-Hot encoding would add hundreds of sparse columns for only 1,143 customers, which increases the risk of overfitting. These features should use Frequency Encoding or have their rare levels grouped. Since they form a geographic hierarchy, keeping only the more general levels may also be enough.

These counts were calculated before handling missing values. Imputing with categories such as `"Unknown"` will add one more level to the affected features.